In [ ]:
import pandas as pd
import numpy as np
import os

# --- PATHS ---
# Update these if your paths are different
BASE_PATH = "/content/drive/MyDrive/1 Skripsi/new approach v1/"

# Input Files
ALPHA_DATA = os.path.join(BASE_PATH, "VPNOnly-cnn_payload_data.npy")
ALPHA_LABELS = os.path.join(BASE_PATH, "VPNOnly-cnn_payload_labels.csv")
BETA_CSV = os.path.join(BASE_PATH, "VPNOnly-delta_component_v2.csv")
GAMMA_CSV = os.path.join(BASE_PATH, "VPNOnly-gamma_prime_component_v2.csv")
FFT_CSV = os.path.join(BASE_PATH, "VPNOnly-fft_component_v2.csv") # <--- NEW INPUT

# Output Files (Overwrites previous versions)
OUTPUT_X_VIEW1 = os.path.join(BASE_PATH, "HMVCL_X_view1_alpha.npy")
OUTPUT_X_VIEW2 = os.path.join(BASE_PATH, "HMVCL_X_view2_stats.npy")
OUTPUT_Y_LABELS = os.path.join(BASE_PATH, "HMVCL_y_labels.csv")

def main():
    print("--- 1. Loading Data Sources ---")

    # --- Load View 1 (Alpha / CNN) ---
    if not os.path.exists(ALPHA_DATA):
        print(f"FATAL: Could not find {ALPHA_DATA}")
        return

    X_alpha = np.load(ALPHA_DATA) # Shape: (N, 10, 784)
    df_alpha = pd.read_csv(ALPHA_LABELS)
    # Add index to track original position so we can grab the right X_alpha rows later
    df_alpha['original_index'] = df_alpha.index
    print(f"Alpha Loaded: {X_alpha.shape}")

    # --- Load View 2 Parts (Beta, Gamma, FFT) ---
    print("Loading Beta, Gamma, and FFT CSVs...")
    df_beta = pd.read_csv(BETA_CSV)
    df_gamma = pd.read_csv(GAMMA_CSV)

    if os.path.exists(FFT_CSV):
        df_fft = pd.read_csv(FFT_CSV)
        print(f"FFT Loaded: {len(df_fft)} rows")
    else:
        print(f"FATAL: Could not find {FFT_CSV}. Run the FFT extraction script first!")
        return

    # --- 2. Merge Statistical Features (View 2) ---
    print("Merging Beta, Gamma, and FFT...")

    # Merge Beta and Gamma on filename
    # Suffixes ensure we don't crash on duplicate columns like 'application'
    df_stats = pd.merge(df_beta, df_gamma, on="filename", suffixes=('_beta', '_gamma'))

    # Merge with FFT
    # FFT columns usually have unique names (e.g., c2s_size_fft_mean), so direct merge is safe
    df_stats = pd.merge(df_stats, df_fft, on="filename", how="inner")

    print(f"Combined Stats Shape: {df_stats.shape}")

    # --- 3. Align View 1 (Images) with View 2 (Stats) ---
    print("Aligning CNN Data with Statistical Data...")

    # We merge on 'filename'.
    # Alpha has 'application', Stats has 'application_beta' etc.
    df_merged = pd.merge(df_alpha, df_stats, on="filename", how="inner")

    print(f"Final Aligned Samples: {len(df_merged)}")

    # --- 4. Construct Final Arrays ---

    # A. View 1: Alpha (Use original indices to fetch from numpy array)
    valid_indices = df_merged['original_index'].values
    X_view1 = X_alpha[valid_indices]

    # B. View 2: Stats (Drop non-numeric cols and label cols)
    # We remove ALL label/metadata columns to leave only features
    drop_cols = [
        'filename', 'application', 'category', 'binary_type', 'original_index',
        'application_beta', 'category_beta', 'binary_type_beta',
        'application_gamma', 'category_gamma', 'binary_type_gamma'
    ]

    # Filter only numeric columns for the MLP input
    # This automatically includes the new FFT columns because they are float64
    numeric_cols = [c for c in df_merged.columns if c not in drop_cols and df_merged[c].dtype in ['float64', 'int64']]

    print(f"Selected {len(numeric_cols)} features for MLP Input (View 2).")
    # print(f"Features: {numeric_cols[:5]} ...") # Uncomment to see feature names

    X_view2 = df_merged[numeric_cols].values

    # C. Labels (Ground Truth comes from Alpha columns)
    # We explicitly select all three hierarchy levels
    # Ensure your alpha_labels.csv or the merged dataframe actually has these columns!

    # If your CSV headers are: filename, application, category, binary_type
    y_labels = df_merged[['filename', 'application', 'category', 'binary_type']]

    print(f"Saving Aligned Labels with columns: {y_labels.columns.tolist()}")
    y_labels.to_csv(OUTPUT_Y_LABELS, index=False)

    # --- 5. Save ---
    print(f"Saving View 1 (CNN Input): {X_view1.shape}")
    np.save(OUTPUT_X_VIEW1, X_view1)

    print(f"Saving View 2 (MLP Input + FFT): {X_view2.shape}")
    np.save(OUTPUT_X_VIEW2, X_view2)

    print(f"Saving Aligned Labels: {len(y_labels)}")
    y_labels.to_csv(OUTPUT_Y_LABELS, index=False)
    print("--- Alignment Complete ---")

if __name__ == "__main__":
    main()

In [ ]:
import pandas as pd
import numpy as np
import os

# --- PATHS ---
# Update these if your paths are different
BASE_PATH = "/content/drive/MyDrive/1 Skripsi/new approach v2/"

# Input Files
ALPHA_DATA = os.path.join(BASE_PATH, "cnn_payload_data.npy")
ALPHA_LABELS = os.path.join(BASE_PATH, "cnn_payload_labels.csv")
BETA_CSV = os.path.join(BASE_PATH, "delta_component_merged_v2.csv")
GAMMA_CSV = os.path.join(BASE_PATH, "gamma_prime_component_v2.csv")
FFT_CSV = os.path.join(BASE_PATH, "fft_component_v2.csv")

# Output Files (Overwrites previous versions)
OUTPUT_X_VIEW1 = os.path.join(BASE_PATH, "FULL_HMVCL_X_view1_alpha.npy")
OUTPUT_X_VIEW2 = os.path.join(BASE_PATH, "FULL_HMVCL_X_view2_stats.npy")
OUTPUT_Y_LABELS = os.path.join(BASE_PATH, "FULL_HMVCL_y_labels.csv")

def main():
    print("--- 1. Loading Data Sources ---")

    # --- Load View 1 (Alpha / CNN) ---
    if not os.path.exists(ALPHA_DATA):
        print(f"FATAL: Could not find {ALPHA_DATA}")
        return

    X_alpha = np.load(ALPHA_DATA) # Shape: (N, 10, 784)
    df_alpha = pd.read_csv(ALPHA_LABELS)
    # Add index to track original position so we can grab the right X_alpha rows later
    df_alpha['original_index'] = df_alpha.index
    print(f"Alpha Loaded: {X_alpha.shape}")

    # --- Load View 2 Parts (Beta, Gamma, FFT) ---
    print("Loading Beta, Gamma, and FFT CSVs...")
    df_beta = pd.read_csv(BETA_CSV)
    df_gamma = pd.read_csv(GAMMA_CSV)

    if os.path.exists(FFT_CSV):
        df_fft = pd.read_csv(FFT_CSV)
        print(f"FFT Loaded: {len(df_fft)} rows")
    else:
        print(f"FATAL: Could not find {FFT_CSV}. Run the FFT extraction script first!")
        return

    # --- 2. Merge Statistical Features (View 2) ---
    print("Merging Beta, Gamma, and FFT...")

    # Merge Beta and Gamma on filename
    # Suffixes ensure we don't crash on duplicate columns like 'application'
    df_stats = pd.merge(df_beta, df_gamma, on="filename", suffixes=('_beta', '_gamma'))

    # Merge with FFT
    # FFT columns usually have unique names (e.g., c2s_size_fft_mean), so direct merge is safe
    df_stats = pd.merge(df_stats, df_fft, on="filename", how="inner")

    print(f"Combined Stats Shape: {df_stats.shape}")

    # --- 3. Align View 1 (Images) with View 2 (Stats) ---
    print("Aligning CNN Data with Statistical Data...")

    # We merge on 'filename'.
    # Alpha has 'application', Stats has 'application_beta' etc.
    df_merged = pd.merge(df_alpha, df_stats, on="filename", how="inner")

    print(f"Final Aligned Samples: {len(df_merged)}")

    # --- 4. Construct Final Arrays ---

    # A. View 1: Alpha (Use original indices to fetch from numpy array)
    valid_indices = df_merged['original_index'].values
    X_view1 = X_alpha[valid_indices]

    # B. View 2: Stats (Drop non-numeric cols and label cols)
    # We remove ALL label/metadata columns to leave only features
    drop_cols = [
        'filename', 'application', 'category', 'binary_type', 'original_index',
        'application_beta', 'category_beta', 'binary_type_beta',
        'application_gamma', 'category_gamma', 'binary_type_gamma'
    ]

    # Filter only numeric columns for the MLP input
    # This automatically includes the new FFT columns because they are float64
    numeric_cols = [c for c in df_merged.columns if c not in drop_cols and df_merged[c].dtype in ['float64', 'int64']]

    print(f"Selected {len(numeric_cols)} features for MLP Input (View 2).")
    # print(f"Features: {numeric_cols[:5]} ...") # Uncomment to see feature names

    X_view2 = df_merged[numeric_cols].values

    # C. Labels (Ground Truth comes from Alpha columns)
    # We explicitly select all three hierarchy levels
    # Ensure your alpha_labels.csv or the merged dataframe actually has these columns!

    # If your CSV headers are: filename, application, category, binary_type
    y_labels = df_merged[['filename', 'application', 'category', 'binary_type']]

    print(f"Saving Aligned Labels with columns: {y_labels.columns.tolist()}")
    y_labels.to_csv(OUTPUT_Y_LABELS, index=False)

    # --- 5. Save ---
    print(f"Saving View 1 (CNN Input): {X_view1.shape}")
    np.save(OUTPUT_X_VIEW1, X_view1)

    print(f"Saving View 2 (MLP Input + FFT): {X_view2.shape}")
    np.save(OUTPUT_X_VIEW2, X_view2)

    print(f"Saving Aligned Labels: {len(y_labels)}")
    y_labels.to_csv(OUTPUT_Y_LABELS, index=False)
    print("--- Alignment Complete ---")

if __name__ == "__main__":
    main()

--- 1. Loading Data Sources ---
Alpha Loaded: (9720, 10, 784)
Loading Beta, Gamma, and FFT CSVs...
FFT Loaded: 13305 rows
Merging Beta, Gamma, and FFT...
Combined Stats Shape: (10105, 107)
Aligning CNN Data with Statistical Data...
Final Aligned Samples: 9542
Selected 100 features for MLP Input (View 2).
Saving Aligned Labels with columns: ['filename', 'application', 'category', 'binary_type']
Saving View 1 (CNN Input): (9542, 10, 784)
Saving View 2 (MLP Input + FFT): (9542, 100)
Saving Aligned Labels: 9542
--- Alignment Complete ---
